# Project 3 — Superstore Commercial Profitability

## Phase 3 — Analysis and Metric Development

### Notebook Purpose

This notebook supports Phase 3 of the Superstore Commercial Profitability mini project.

The purpose is to:

- define and validate business metrics;
- calculate commercial profitability summaries;
- analyze sales, profit, weighted margin and loss-making records;
- evaluate product category and sub-category contribution;
- evaluate discount-tier profitability;
- review regional and state-level contribution;
- analyze historical customer contribution and Pareto concentration;
- identify candidate findings for Tableau dashboarding.

This notebook uses the SQL-staged table:

`raw_superstore_sales`

from the SQLite database:

`data/database/superstore_commercial_analytics.db`

---

## Phase 3 Guardrails

This notebook should not:

- claim discounts caused losses;
- call historical customer contribution true customer lifetime value;
- reconstruct gross or net revenue without source fields;
- rely on product_name as a unique product key;
- expand into basket analysis before core profitability analysis is complete;
- build dashboard visuals before metrics are validated.

---

## Confirmed Phase 2 Context

Phase 2 confirmed:

- 9,994 line-item rows
- 5,009 distinct orders
- 793 distinct customers
- 1,862 distinct products
- Order Date range: 2014-01-03 to 2017-12-30
- Ship Date range: 2014-01-07 to 2018-01-05
- 1,871 negative-profit line items
- 0 sales less than or equal to zero
- 0 records where Ship Date is earlier than Order Date

Confirmed grain:

One row represents one product line item within an order.

Default business date:

Use `order_date` for sales, profit, margin and year-over-year analysis.

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

project_root = Path.cwd().parent
db_path = project_root / "data" / "database" / "superstore_commercial_analytics.db"

print("Project root:", project_root)
print("Database path:", db_path)

def run_sql(query):
    """
    Run a SQL query against the Superstore SQLite database
    and return the result as a pandas DataFrame.
    """
    conn = sqlite3.connect(db_path)
    try:
        result = pd.read_sql_query(query, conn)
    finally:
        conn.close()
    return result

Project root: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics
Database path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\data\database\superstore_commercial_analytics.db


## 1. Overall Commercial Summary

### Purpose

Create the project baseline metrics for sales, profit, margin, order count, customer count, product count and loss-making records.

### Why this matters

This summary establishes the commercial size and profitability profile of the dataset before drilling into category, discount, region or customer-level patterns.

### Grain caution

The table has line-item grain. Therefore:

- `COUNT(*)` means line-item count.
- `COUNT(DISTINCT order_id)` means order count.
- `COUNT(DISTINCT customer_id)` means customer count.
- `COUNT(DISTINCT product_id)` means product count.

### Metric caution

Use weighted profit margin:

`SUM(profit) / SUM(sales)`

Do not use average row-level margin as the primary business margin.

In [7]:
overall_summary = run_sql("""
SELECT
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT order_id) AS order_count,
    COUNT(DISTINCT customer_id) AS customer_count,
    COUNT(DISTINCT product_id) AS product_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_line_items,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_line_item_share,
    ROUND(SUM(CASE WHEN profit < 0 THEN sales ELSE 0 END), 2) AS loss_making_sales,
    ROUND(SUM(CASE WHEN profit < 0 THEN profit ELSE 0 END), 2) AS loss_making_profit
FROM raw_superstore_sales;
""")

overall_summary

,line_item_count,order_count,customer_count,product_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share,loss_making_sales,loss_making_profit
0,9994,5009,793,1862,2297200.86,286397.02,0.1247,1871,0.1872,468707.15,-156131.29


# Phase 3 Result — Overall Commercial Summary

Result:
- Line item count: 9,994
- Order count: 5,009
- Customer count: 793
- Product count: 1,862
- Total sales: 2,297,200.86
- Total profit: 286,397.02
- Weighted profit margin: 12.47%
- Loss-making line items: 1,871
- Loss-making line-item share: 18.72%
- Loss-making sales: 468,707.15
- Loss-making profit: -156,131.29

Interpretation:
The staged Superstore dataset contains 2.30M in sales and 286.4K in profit, with an overall weighted profit margin of 12.47%. Loss-making line items represent 18.72% of records and are associated with 468.7K in sales and -156.1K in profit.

Decision:
The overall commercial baseline supports a profitability review focused on revenue leakage. Phase 3 should investigate where loss-making sales and weak margin concentrate across discount tier, category, sub-category, region, and customer contribution.

Caution:
The presence of loss-making line items does not explain cause. Discount, product, and region patterns should be described as associations unless stronger evidence is available.

In [8]:
category_summary = run_sql("""
SELECT
    category,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_line_items,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_line_item_share,
    ROUND(SUM(CASE WHEN profit < 0 THEN sales ELSE 0 END), 2) AS loss_making_sales,
    ROUND(SUM(CASE WHEN profit < 0 THEN profit ELSE 0 END), 2) AS loss_making_profit
FROM raw_superstore_sales
GROUP BY category
ORDER BY total_sales DESC;
""")

category_summary

,category,line_item_count,order_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share,loss_making_sales,loss_making_profit
0,Technology,1847,1544,836154.03,145454.95,0.1740,271,0.1467,119212.89,-38579.92
1,Furniture,2121,1764,741999.80,18451.27,0.0249,714,0.3366,257885.59,-60936.11
2,Office Supplies,6026,3742,719047.03,122490.80,0.1704,886,0.1470,91608.68,-56615.26


# Phase 3 Result — Category Profitability Summary

Result:
Technology:
- Sales: 836,154.03
- Profit: 145,454.95
- Weighted profit margin: 17.40%
- Loss-making line-item share: 14.67%
- Loss-making sales: 119,212.89
- Loss-making profit: -38,579.92

Furniture:
- Sales: 741,999.80
- Profit: 18,451.27
- Weighted profit margin: 2.49%
- Loss-making line-item share: 33.66%
- Loss-making sales: 257,885.59
- Loss-making profit: -60,936.11

Office Supplies:
- Sales: 719,047.03
- Profit: 122,490.80
- Weighted profit margin: 17.04%
- Loss-making line-item share: 14.70%
- Loss-making sales: 91,608.68
- Loss-making profit: -56,615.26

Interpretation:
Furniture generated substantial sales volume but much weaker profit conversion than Technology and Office Supplies. Furniture had a 2.49% weighted profit margin and a 33.66% loss-making line-item share, compared with margins above 17% for Technology and Office Supplies.

Decision:
Category analysis is strong enough to carry into the final dashboard story. Furniture should be reviewed further at the sub-category and discount-tier levels.

Caution:
This result shows an observed profitability pattern. It does not prove the cause of weak Furniture profitability.

In [9]:
sub_category_summary = run_sql("""
SELECT
    category,
    sub_category,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_line_items,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_line_item_share,
    ROUND(SUM(CASE WHEN profit < 0 THEN sales ELSE 0 END), 2) AS loss_making_sales,
    ROUND(SUM(CASE WHEN profit < 0 THEN profit ELSE 0 END), 2) AS loss_making_profit
FROM raw_superstore_sales
GROUP BY category, sub_category
ORDER BY total_profit ASC;
""")

sub_category_summary

,category,sub_category,line_item_count,order_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share,loss_making_sales,loss_making_profit
0,Furniture,Tables,319,307,206965.53,-17725.48,-0.0856,203,0.6364,104978.55,-32412.15
1,Furniture,Bookcases,228,224,114880.00,-3472.56,-0.0302,109,0.4781,48072.74,-12152.21
2,Office Supplies,Supplies,190,187,46673.54,-1189.10,-0.0255,33,0.1737,14067.18,-3015.62
3,Office Supplies,Fasteners,217,215,3024.28,949.52,0.3140,12,0.0553,149.28,-33.20
4,Technology,Machines,115,112,189238.63,3384.76,0.0179,44,0.3826,72456.25,-30118.67
5,Office Supplies,Labels,364,346,12486.31,5546.25,0.4442,0,0.0000,0.00,0.00
6,Office Supplies,Art,796,731,27118.79,6527.79,0.2407,0,0.0000,0.00,0.00
7,Office Supplies,Envelopes,254,249,16476.40,6964.18,0.4227,0,0.0000,0.00,0.00
8,Furniture,Furnishings,957,877,91705.16,13059.14,0.1424,167,0.1745,12845.84,-6490.91
9,Office Supplies,Appliances,466,451,107532.16,18138.01,0.1687,67,0.1438,3382.53,-8629.64


# Phase 3 Result — Sub-Category Profitability Summary

Result:
Sub-category analysis shows that category-level profitability differences are concentrated in specific sub-categories.

Key weak-profit sub-categories:
- Tables: 206,965.53 sales, -17,725.48 profit, -8.56% weighted margin, 63.64% loss-making line-item share
- Bookcases: 114,880.00 sales, -3,472.56 profit, -3.02% weighted margin, 47.81% loss-making line-item share
- Supplies: 46,673.54 sales, -1,189.10 profit, -2.55% weighted margin, 17.37% loss-making line-item share

High-sales / weak-profit review area:
- Machines: 189,238.63 sales, 3,384.76 profit, 1.79% weighted margin, 38.26% loss-making line-item share

Important mixed-performance area:
- Binders: 203,412.73 sales, 30,221.76 profit, 14.86% weighted margin, 40.25% loss-making line-item share and -38,510.50 loss-making profit

Strong contrast sub-categories:
- Copiers: 149,528.03 sales, 55,617.82 profit, 37.20% weighted margin, 0.00% loss-making line-item share
- Paper: 78,479.21 sales, 34,053.57 profit, 43.39% weighted margin, 0.00% loss-making line-item share
- Labels: 12,486.31 sales, 5,546.25 profit, 44.42% weighted margin, 0.00% loss-making line-item share
- Envelopes: 16,476.40 sales, 6,964.18 profit, 42.27% weighted margin, 0.00% loss-making line-item share

Interpretation:
Furniture’s weak category profitability is concentrated mainly in Tables and Bookcases rather than all Furniture sub-categories. Machines also appear as a high-sales but weak-profit Technology sub-category. Binders are profitable overall but have a large loss-making line-item share, making them worth reviewing by discount tier or region.

Decision:
Sub-category analysis is strong enough to carry into the final dashboard. The dashboard should prioritize category and sub-category profitability rather than product-name-level rankings.

Caution:
These results show observed profitability patterns. They do not prove why losses occurred.

In [10]:
discount_category_summary = run_sql("""
SELECT
    category,
    discount_tier,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_line_items,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_line_item_share,
    ROUND(SUM(CASE WHEN profit < 0 THEN sales ELSE 0 END), 2) AS loss_making_sales,
    ROUND(SUM(CASE WHEN profit < 0 THEN profit ELSE 0 END), 2) AS loss_making_profit
FROM raw_superstore_sales
GROUP BY category, discount_tier
ORDER BY category,
    CASE discount_tier
        WHEN 'No Discount' THEN 1
        WHEN 'Low Discount' THEN 2
        WHEN 'Moderate Discount' THEN 3
        WHEN 'High Discount' THEN 4
        ELSE 5
    END;
""")

discount_category_summary

,category,discount_tier,line_item_count,order_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share,loss_making_sales,loss_making_profit
0,Furniture,No Discount,836,739,256025.27,58133.08,0.2271,0,0.0000,0.00,0.00
1,Furniture,Low Discount,128,122,74192.77,8530.00,0.1150,21,0.1641,7387.50,-357.98
2,Furniture,Moderate Discount,837,728,316101.37,-4429.37,-0.0140,373,0.4456,154817.70,-16795.69
3,Furniture,High Discount,320,296,95680.39,-43782.44,-0.4576,320,1.0000,95680.39,-43782.44
4,Office Supplies,No Discount,3129,2020,442150.00,130506.11,0.2952,0,0.0000,0.00,0.00
5,Office Supplies,Low Discount,16,16,4324.15,1086.08,0.2512,0,0.0000,0.00,0.00
6,Office Supplies,Moderate Discount,2201,1661,233049.74,38038.75,0.1632,206,0.0936,52085.53,-9475.12
7,Office Supplies,High Discount,680,573,39523.15,-47140.14,-1.1927,680,1.0000,39523.15,-47140.14
8,Technology,No Discount,833,718,389733.20,132348.42,0.3396,0,0.0000,0.00,0.00
9,Technology,Low Discount,2,2,3410.95,832.08,0.2439,0,0.0000,0.00,0.00


# Phase 3 Result — Discount Tier by Category

Result:
High-discount line items show negative aggregate profit across all three major product categories.

Furniture:
- No Discount: 256,025.27 sales, 58,133.08 profit, 22.71% weighted margin, 0.00% loss-making share
- Low Discount: 74,192.77 sales, 8,530.00 profit, 11.50% weighted margin, 16.41% loss-making share
- Moderate Discount: 316,101.37 sales, -4,429.37 profit, -1.40% weighted margin, 44.56% loss-making share
- High Discount: 95,680.39 sales, -43,782.44 profit, -45.76% weighted margin, 100.00% loss-making share

Office Supplies:
- No Discount: 442,150.00 sales, 130,506.11 profit, 29.52% weighted margin, 0.00% loss-making share
- Low Discount: 4,324.15 sales, 1,086.08 profit, 25.12% weighted margin, 0.00% loss-making share
- Moderate Discount: 233,049.74 sales, 38,038.75 profit, 16.32% weighted margin, 9.36% loss-making share
- High Discount: 39,523.15 sales, -47,140.14 profit, -119.27% weighted margin, 100.00% loss-making share

Technology:
- No Discount: 389,733.20 sales, 132,348.42 profit, 33.96% weighted margin, 0.00% loss-making share
- Low Discount: 3,410.95 sales, 832.08 profit, 24.39% weighted margin, 0.00% loss-making share
- Moderate Discount: 318,669.92 sales, 46,358.65 profit, 14.55% weighted margin, 15.48% loss-making share
- High Discount: 124,339.96 sales, -34,084.20 profit, -27.41% weighted margin, 84.34% loss-making share

Interpretation:
Discount-tier analysis is highly relevant. High-discount rows are associated with negative aggregate profit in all three categories. Furniture and Office Supplies show especially severe high-discount pressure, with 100% of high-discount line items loss-making in both categories. Furniture also turns negative at the moderate discount tier.

Decision:
Discount-tier profitability should be a core dashboard and findings component. The final project should include a discount-tier profitability view by category.

Caution:
The observed relationship between discount tier and profitability is associative. The data does not prove that discounts caused losses or that removing discounts would recover profit.

In [11]:
subcat_discount_summary = run_sql("""
SELECT
    category,
    sub_category,
    discount_tier,
    COUNT(*) AS line_item_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_line_items,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_line_item_share
FROM raw_superstore_sales
GROUP BY category, sub_category, discount_tier
HAVING SUM(sales) >= 10000
ORDER BY total_profit ASC;
""")

subcat_discount_summary

,category,sub_category,discount_tier,line_item_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share
0,Office Supplies,Binders,High Discount,613,36140.61,-38510.50,-1.0656,613,1.0000
1,Technology,Machines,High Discount,48,73082.80,-29881.39,-0.4089,43,0.8958
2,Furniture,Tables,High Discount,122,64774.39,-27295.90,-0.4214,122,1.0000
3,Furniture,Bookcases,High Discount,60,24261.30,-10541.89,-0.4345,60,1.0000
4,Technology,Phones,High Discount,109,34337.35,-6385.79,-0.1860,97,0.8899
5,Office Supplies,Storage,Moderate Discount,316,65989.85,-4249.35,-0.0644,161,0.5095
6,Furniture,Tables,Moderate Discount,125,70612.38,-3705.89,-0.0525,81,0.6480
7,Office Supplies,Supplies,Moderate Discount,73,15114.33,-2907.49,-0.1924,33,0.4521
8,Furniture,Chairs,Moderate Discount,408,190754.13,-2453.94,-0.0129,231,0.5662
9,Furniture,Bookcases,Moderate Discount,56,31124.19,-425.37,-0.0137,32,0.5714


# Phase 3 Result — Sub-Category by Discount Tier

Result:
Sub-category by discount-tier analysis shows that profit leakage is concentrated in specific product and discount combinations rather than uniformly across entire categories.

Largest negative-profit combinations among combinations with at least 10,000 in sales:
- Binders / High Discount: 36,140.61 sales, -38,510.50 profit, -106.56% weighted margin, 100.00% loss-making line-item share
- Machines / High Discount: 73,082.80 sales, -29,881.39 profit, -40.89% weighted margin, 89.58% loss-making line-item share
- Tables / High Discount: 64,774.39 sales, -27,295.90 profit, -42.14% weighted margin, 100.00% loss-making line-item share
- Bookcases / High Discount: 24,261.30 sales, -10,541.89 profit, -43.45% weighted margin, 100.00% loss-making line-item share
- Phones / High Discount: 34,337.35 sales, -6,385.79 profit, -18.60% weighted margin, 88.99% loss-making line-item share

Important nuance:
Some sub-categories are profitable at no discount but weak or negative under higher discount tiers. Tables, Machines and Binders show especially clear tier-based profitability differences.

Interpretation:
The strongest commercial review areas are not entire categories but specific sub-category and discount-tier combinations. High-discount Binders, Machines, Tables, Bookcases and Phones should be prioritized for review.

Decision:
The final dashboard should include a Sub-Category Profitability Review Matrix using sub-category and discount tier. This visual is more useful than a simple category bar chart because it identifies where profit leakage is concentrated.

Caution:
Discount tier is associated with weaker profitability, but the dataset does not identify why discounts were applied. Possible business explanations could include clearance, weak demand, damaged goods, missing parts, quality issues or promotions, but these are hypotheses only and cannot be confirmed from the available fields.

In [12]:
region_summary = run_sql("""
SELECT
    region,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT order_id) AS order_count,
    COUNT(DISTINCT customer_id) AS customer_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_line_items,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_line_item_share,
    ROUND(SUM(CASE WHEN profit < 0 THEN sales ELSE 0 END), 2) AS loss_making_sales,
    ROUND(SUM(CASE WHEN profit < 0 THEN profit ELSE 0 END), 2) AS loss_making_profit
FROM raw_superstore_sales
GROUP BY region
ORDER BY total_sales DESC;
""")

region_summary

,region,line_item_count,order_count,customer_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share,loss_making_sales,loss_making_profit
0,West,3203,1611,686,725457.82,108418.45,0.1494,318,0.0993,74925.30,-22720.96
1,East,2848,1401,674,678781.24,91522.78,0.1348,553,0.1942,160864.01,-49590.61
2,Central,2323,1175,629,501239.89,39706.36,0.0792,741,0.3190,141282.66,-56314.89
3,South,1620,822,512,391721.91,46749.43,0.1193,259,0.1599,91635.18,-27504.83


# Phase 3 Result — Region Profitability Summary

Result:
- West: 725,457.82 sales, 108,418.45 profit, 14.94% weighted margin, 9.93% loss-making line-item share, 74,925.30 loss-making sales, -22,720.96 loss-making profit
- East: 678,781.24 sales, 91,522.78 profit, 13.48% weighted margin, 19.42% loss-making line-item share, 160,864.01 loss-making sales, -49,590.61 loss-making profit
- Central: 501,239.89 sales, 39,706.36 profit, 7.92% weighted margin, 31.90% loss-making line-item share, 141,282.66 loss-making sales, -56,314.89 loss-making profit
- South: 391,721.91 sales, 46,749.43 profit, 11.93% weighted margin, 15.99% loss-making line-item share, 91,635.18 loss-making sales, -27,504.83 loss-making profit

Interpretation:
Regional analysis is useful as a supporting lens. Central has the weakest regional profit profile, with the lowest weighted margin and the highest loss-making line-item share. West is the strongest region by total sales, total profit and low loss-making exposure.

Decision:
Use Region as a supporting dashboard filter or secondary view. The main dashboard story should remain focused on sub-category and discount-tier profitability, with region used to identify where weak-margin or loss-making patterns are geographically concentrated.

Caution:
Regional patterns are descriptive. The dataset does not explain why specific regions have weaker profitability.

# Scope Note — Logistics and Supply Route Analysis

The Superstore dataset supports commercial profitability analysis by order geography, product category, customer segment, discount tier, sales and profit.

It does not support supply-chain routing or product shipment origin analysis.

Unavailable fields include:
- warehouse or fulfillment center location
- supplier location
- port of entry
- carrier route
- shipping cost
- distance traveled
- inventory location
- product condition
- returns or damage reason
- fulfillment path

Because of these missing fields, the project cannot determine whether products entered through the West Coast, moved through southern land routes, or followed any specific logistics network.

Region and state can be used to analyze customer/order geography and profitability patterns, but not physical product movement or supply-chain routing.

Decision:
Keep logistics routing out of scope for this mini project. Treat region/state as commercial geography only.

In [13]:
customer_summary = run_sql("""
SELECT
    customer_id,
    customer_name,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_line_items,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_line_item_share
FROM raw_superstore_sales
GROUP BY customer_id, customer_name
ORDER BY total_sales DESC
LIMIT 20;
""")

customer_summary

,customer_id,customer_name,line_item_count,order_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share
0,SM-20320,Sean Miller,15,5,25043.05,-1980.74,-0.0791,5,0.3333
1,TC-20980,Tamara Chand,12,5,19052.22,8981.32,0.4714,1,0.0833
2,RB-19360,Raymond Buch,18,6,15117.34,6976.10,0.4615,1,0.0556
3,TA-21385,Tom Ashbrook,10,4,14595.62,4703.79,0.3223,1,0.1000
4,AB-10105,Adrian Barton,20,10,14473.57,5444.81,0.3762,7,0.3500
5,KL-16645,Ken Lonsdale,29,12,14175.23,806.85,0.0569,7,0.2414
6,SC-20095,Sanjit Chand,22,9,14142.33,5757.41,0.4071,2,0.0909
7,HL-15040,Hunter Lopez,11,6,12873.30,5622.43,0.4368,0,0.0000
8,SE-20110,Sanjit Engle,19,11,12209.44,2650.68,0.2171,3,0.1579
9,CC-12370,Christopher Conant,11,5,12129.07,2177.05,0.1795,4,0.3636


# Phase 3 Result — Top Customers by Sales

Result:
The top 20 customers by sales show that sales contribution and profit contribution are not always aligned.

Examples:
- Sean Miller generated 25,043.05 in sales but -1,980.74 in profit, with a -7.91% weighted margin.
- Tamara Chand generated 19,052.22 in sales and 8,981.32 in profit, with a 47.14% weighted margin.
- Raymond Buch generated 15,117.34 in sales and 6,976.10 in profit, with a 46.15% weighted margin.
- Becky Martin generated 11,789.63 in sales but -1,659.96 in profit, with a -14.08% weighted margin.
- Ken Lonsdale generated 14,175.23 in sales but only 806.85 in profit, with a 5.69% weighted margin.

Interpretation:
High customer sales do not necessarily translate into high profit. Customer-level contribution is useful for identifying revenue concentration and profit-quality differences, but it should remain a supporting analytical layer.

Decision:
Customer contribution analysis is feasible and should be included as a supporting Phase 3 output. It should not become the main dashboard story unless Pareto analysis shows strong concentration.

Caution:
This is historical customer contribution, not true customer lifetime value. The dataset does not include acquisition cost, retention horizon, future value or full customer lifecycle data.

In [14]:
customer_pareto = run_sql("""
WITH customer_totals AS (
    SELECT
        customer_id,
        customer_name,
        ROUND(SUM(sales), 2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit
    FROM raw_superstore_sales
    GROUP BY customer_id, customer_name
),

ranked_customers AS (
    SELECT
        customer_id,
        customer_name,
        total_sales,
        total_profit,
        ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS sales_rank,
        COUNT(*) OVER () AS total_customers,
        SUM(total_sales) OVER () AS grand_total_sales,
        SUM(total_profit) OVER () AS grand_total_profit,
        SUM(total_sales) OVER (
            ORDER BY total_sales DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_sales,
        SUM(total_profit) OVER (
            ORDER BY total_sales DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_profit
    FROM customer_totals
)

SELECT
    customer_id,
    customer_name,
    sales_rank,
    total_customers,
    total_sales,
    total_profit,
    ROUND(1.0 * sales_rank / total_customers, 4) AS customer_rank_share,
    ROUND(cumulative_sales, 2) AS cumulative_sales,
    ROUND(1.0 * cumulative_sales / grand_total_sales, 4) AS cumulative_sales_share,
    ROUND(cumulative_profit, 2) AS cumulative_profit,
    ROUND(1.0 * cumulative_profit / grand_total_profit, 4) AS cumulative_profit_share
FROM ranked_customers
WHERE sales_rank <= 20
ORDER BY sales_rank;
""")

customer_pareto

,customer_id,customer_name,sales_rank,total_customers,total_sales,total_profit,customer_rank_share,cumulative_sales,cumulative_sales_share,cumulative_profit,cumulative_profit_share
0,SM-20320,Sean Miller,1,793,25043.05,-1980.74,0.0013,25043.05,0.0109,-1980.74,-0.0069
1,TC-20980,Tamara Chand,2,793,19052.22,8981.32,0.0025,44095.27,0.0192,7000.58,0.0244
2,RB-19360,Raymond Buch,3,793,15117.34,6976.10,0.0038,59212.61,0.0258,13976.68,0.0488
3,TA-21385,Tom Ashbrook,4,793,14595.62,4703.79,0.0050,73808.23,0.0321,18680.47,0.0652
4,AB-10105,Adrian Barton,5,793,14473.57,5444.81,0.0063,88281.80,0.0384,24125.28,0.0842
5,KL-16645,Ken Lonsdale,6,793,14175.23,806.85,0.0076,102457.03,0.0446,24932.13,0.0871
6,SC-20095,Sanjit Chand,7,793,14142.33,5757.41,0.0088,116599.36,0.0508,30689.54,0.1072
7,HL-15040,Hunter Lopez,8,793,12873.30,5622.43,0.0101,129472.66,0.0564,36311.97,0.1268
8,SE-20110,Sanjit Engle,9,793,12209.44,2650.68,0.0113,141682.10,0.0617,38962.65,0.1360
9,CC-12370,Christopher Conant,10,793,12129.07,2177.05,0.0126,153811.17,0.0670,41139.70,0.1436


# Phase 3 Result — Customer Pareto Top 20 by Sales

Result:
The top 20 customers by sales represent 2.52% of customers and account for 11.53% of total sales and 19.39% of total profit.

Interpretation:
Customer contribution is somewhat concentrated, but not extremely concentrated among the top 20 customers. High sales contribution does not always translate into high profit contribution. The highest-sales customer, Sean Miller, generated 25,043.05 in sales but -1,980.74 in profit. Other high-sales customers, such as Tamara Chand and Raymond Buch, generated strong profit.

Decision:
Customer contribution and Pareto analysis are useful as a supporting analytical layer. They should not replace the main dashboard story around sub-category and discount-tier profitability.

Caution:
This is historical customer contribution, not true customer lifetime value. The dataset does not include acquisition cost, retention horizon, future value or full lifecycle fields.

In [15]:
customer_top20_share = run_sql("""
WITH customer_totals AS (
    SELECT
        customer_id,
        customer_name,
        SUM(sales) AS total_sales,
        SUM(profit) AS total_profit
    FROM raw_superstore_sales
    GROUP BY customer_id, customer_name
),

ranked_customers AS (
    SELECT
        customer_id,
        customer_name,
        total_sales,
        total_profit,
        ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS sales_rank,
        COUNT(*) OVER () AS total_customers,
        SUM(total_sales) OVER () AS grand_total_sales,
        SUM(total_profit) OVER () AS grand_total_profit
    FROM customer_totals
)

SELECT
    COUNT(*) AS top_customer_count,
    MAX(total_customers) AS total_customers,
    ROUND(1.0 * COUNT(*) / MAX(total_customers), 4) AS customer_share,
    ROUND(SUM(total_sales), 2) AS top_customer_sales,
    ROUND(1.0 * SUM(total_sales) / MAX(grand_total_sales), 4) AS top_customer_sales_share,
    ROUND(SUM(total_profit), 2) AS top_customer_profit,
    ROUND(1.0 * SUM(total_profit) / MAX(grand_total_profit), 4) AS top_customer_profit_share
FROM ranked_customers
WHERE sales_rank <= CAST(total_customers * 0.20 AS INTEGER);
""")

customer_top20_share

,top_customer_count,total_customers,customer_share,top_customer_sales,top_customer_sales_share,top_customer_profit,top_customer_profit_share
0,158,793,0.1992,1101781.39,0.4796,169872.62,0.5931


# Phase 3 Result — Top 20% Customer Contribution

Result:
- Top customer count: 158
- Total customers: 793
- Customer share: 19.92%
- Top customer sales: 1,101,781.39
- Top customer sales share: 47.96%
- Top customer profit: 169,872.62
- Top customer profit share: 59.31%

Interpretation:
The top approximately 20% of customers account for 47.96% of total sales and 59.31% of total profit. Customer contribution is meaningfully concentrated, especially for profit.

Decision:
Customer Pareto analysis should be included as a supporting insight, but it should not replace the main dashboard story around sub-category and discount-tier profitability.

Caution:
This is historical customer contribution, not true customer lifetime value. The result should not be described as CLV or used to infer future customer value without additional lifecycle, retention and acquisition cost data.

In [16]:
yoy_summary = run_sql("""
SELECT
    order_year,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT order_id) AS order_count,
    COUNT(DISTINCT customer_id) AS customer_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_line_items,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_line_item_share
FROM raw_superstore_sales
GROUP BY order_year
ORDER BY order_year;
""")

yoy_summary

,order_year,line_item_count,order_count,customer_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share
0,2014,1993,969,595,484247.50,49543.97,0.1023,377,0.1892
1,2015,2102,1038,573,470532.51,61618.60,0.1310,397,0.1889
2,2016,2587,1315,638,609205.60,81795.17,0.1343,477,0.1844
3,2017,3312,1687,693,733215.26,93439.27,0.1274,620,0.1872


# Phase 3 Result — Year-over-Year Summary

Result:
2014:
- Sales: 484,247.50
- Profit: 49,543.97
- Weighted profit margin: 10.23%
- Orders: 969
- Loss-making line-item share: 18.92%

2015:
- Sales: 470,532.51
- Profit: 61,618.60
- Weighted profit margin: 13.10%
- Orders: 1,038
- Loss-making line-item share: 18.89%

2016:
- Sales: 609,205.60
- Profit: 81,795.17
- Weighted profit margin: 13.43%
- Orders: 1,315
- Loss-making line-item share: 18.44%

2017:
- Sales: 733,215.26
- Profit: 93,439.27
- Weighted profit margin: 12.74%
- Orders: 1,687
- Loss-making line-item share: 18.72%

Interpretation:
Sales and profit increased over the 2014–2017 period, while weighted profit margin stayed around 13% after 2014. Loss-making line-item share remained stable at approximately 18–19% each year, suggesting that loss-making activity is persistent rather than isolated to a single year.

Decision:
YoY analysis is useful as supporting context, but it should not replace the main dashboard story. The strongest project story remains sub-category and discount-tier profitability.

Caution:
YoY growth may reflect increased order activity and line-item volume, not necessarily improved commercial efficiency.

# Phase 4 Candidate Dashboard Visuals

## Dashboard Purpose

The dashboard should help a Commercial Analytics Manager or VP of Sales Operations identify where sales volume is not translating into proportional profit and where commercial review should be prioritized.

## Primary Dashboard Story

Profit leakage is concentrated in specific sub-category and discount-tier combinations, especially high-discount Binders, Machines, Tables, Bookcases and Phones.

## Recommended Dashboard Visuals

### 1. KPI Header

Purpose:
Show the overall commercial baseline.

Candidate KPI cards:
- Total Sales
- Total Profit
- Weighted Profit Margin
- Order Count
- Loss-Making Line-Item Share
- Loss-Making Sales

Rationale:
These metrics summarize project scale and establish why profitability review is needed.

---

### 2. Sub-Category Profitability Review Matrix

Purpose:
Identify which sub-category and discount-tier combinations should be prioritized for commercial review.

Recommended structure:
- Rows: Sub-Category
- Columns: Discount Tier
- Color: Weighted Profit Margin or Total Profit
- Tooltip: Sales, Profit, Line Item Count, Loss-Making Line-Item Share
- Filters: Category, Region, Segment

Rationale:
This is the strongest visual because it shows that weak profitability is concentrated in specific product and discount combinations rather than entire categories.

---

### 3. Sub-Category Sales vs Weighted Margin View

Purpose:
Identify high-sales, weak-margin sub-categories.

Recommended structure:
- x-axis: Total Sales
- y-axis: Weighted Profit Margin
- color: Category
- label: Sub-Category
- tooltip: Profit, Loss-Making Share, Discount Tier context

Rationale:
This helps stakeholders identify sub-categories where revenue volume does not translate into profit.

---

### 4. Regional Profitability Support View

Purpose:
Show whether weak-margin or loss-making patterns have geographic concentration.

Recommended structure:
- Region-level bar chart or summary table
- Metrics: Sales, Profit, Weighted Profit Margin, Loss-Making Line-Item Share
- Optional State detail in tooltip or drill-down

Rationale:
Region is useful as a supporting lens. Central has the weakest regional profit profile, but geography should not replace the main product/discount story.

---

### Optional 5. Customer Pareto View

Purpose:
Show historical customer contribution concentration.

Candidate metric:
Top approximately 20% of customers account for 47.96% of sales and 59.31% of profit.

Rationale:
Customer Pareto is useful as a supporting insight, but it should not dominate the dashboard.

## Visuals Not Prioritized

Do not prioritize:
- product-name-only rankings
- basket analysis
- shipping-mode analysis
- country-level geography
- forecasting
- causal discount impact visuals
- gross/net revenue visuals

Reason:
These are either unsupported by the dataset, outside mini-project scope, or weaker than the product/discount profitability story.

## Phase 4 Preparation — Export Dashboard-Ready Tables

### Purpose

Export validated Phase 3 analysis outputs as CSV files for Tableau dashboarding.

### Why this matters

Tableau should use clean, analysis-ready summary tables rather than rebuilding every metric manually from the raw line-item table. This reduces metric drift and keeps dashboard calculations aligned with SQL-validated results.

### Export location

Dashboard-ready CSV files will be saved to:

`outputs/dashboard_data/`

### Exported tables

- overall_commercial_summary.csv
- category_summary.csv
- sub_category_summary.csv
- discount_category_summary.csv
- subcat_discount_summary.csv
- region_summary.csv
- customer_top20_share.csv
- yoy_summary.csv

### Caveat

These exports are generated artifacts and can be regenerated from the staged SQLite table and Phase 3 notebook.

In [17]:
from pathlib import Path

dashboard_data_path = project_root / "outputs" / "dashboard_data"
dashboard_data_path.mkdir(parents=True, exist_ok=True)

exports = {
    "overall_commercial_summary.csv": overall_summary,
    "category_summary.csv": category_summary,
    "sub_category_summary.csv": sub_category_summary,
    "discount_category_summary.csv": discount_category_summary,
    "subcat_discount_summary.csv": subcat_discount_summary,
    "region_summary.csv": region_summary,
    "customer_top20_share.csv": customer_top20_share,
    "yoy_summary.csv": yoy_summary
}

for filename, dataframe in exports.items():
    output_path = dashboard_data_path / filename
    dataframe.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

Saved: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\outputs\dashboard_data\overall_commercial_summary.csv
Saved: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\outputs\dashboard_data\category_summary.csv
Saved: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\outputs\dashboard_data\sub_category_summary.csv
Saved: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\outputs\dashboard_data\discount_category_summary.csv
Saved: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\outputs\dashboard_data\subcat_discount_summary.csv
Saved: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\outputs\dashboard_data\region_summary.csv
Saved: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\outputs\dashboard_data\customer_top20_share.csv
Saved: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\outputs\dashboard_data\yoy_summary.csv


In [19]:
print("Project root:", project_root)
print("Database path:", db_path)

sql_script_path = project_root / "sql" / "04_analytical_table.sql"
print("SQL script path:", sql_script_path)
print("SQL script exists:", sql_script_path.exists())

if sql_script_path.exists():
    print("SQL script size:", sql_script_path.stat().st_size, "bytes")

Project root: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics
Database path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\data\database\superstore_commercial_analytics.db
SQL script path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\sql\04_analytical_table.sql
SQL script exists: True
SQL script size: 5664 bytes


In [22]:
sql_script_path = project_root / "sql" / "04_analytical_table.sql"

sql_script = sql_script_path.read_text(encoding="utf-8")

conn = sqlite3.connect(db_path)

try:
    conn.executescript(sql_script)
    conn.commit()
    print("Analytical views created successfully.")
finally:
    conn.close()

Analytical views created successfully.


In [23]:
run_sql("""
SELECT *
FROM vw_overall_commercial_summary;
""")

,line_item_count,order_count,customer_count,product_count,total_sales,total_profit,weighted_profit_margin,loss_making_line_items,loss_making_line_item_share,loss_making_sales,loss_making_profit
0,9994,5009,793,1862,2297200.86,286397.02,0.1247,1871,0.1872,468707.15,-156131.29


In [24]:
run_sql("""
SELECT
    name,
    type
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name;
""")

,name,type
0,raw_superstore_sales,table
1,vw_category_summary,view
2,vw_customer_pareto,view
3,vw_customer_summary,view
4,vw_discount_category_summary,view
5,vw_overall_commercial_summary,view
6,vw_region_summary,view
7,vw_sub_category_summary,view
8,vw_subcat_discount_summary,view
9,vw_yoy_summary,view
